# 02 — Baseline Model: Popularity Recommender

This notebook demonstrates the popularity baseline:
- Load preprocessed train/test splits
- Fit PopularityRecommender
- Evaluate P@K, HR@K, NDCG@K
- Visualise recommendation quality

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import PROCESSED_DIR
from src.models.popularity import PopularityRecommender
from src.evaluation.benchmark import build_ground_truth, build_user_seen_items
from src.evaluation.metrics import compute_ranking_metrics
from src.logging_utils import model_logger

sns.set_style('whitegrid')
model_logger.start_phase('baseline_nb', 'Baseline notebook started')

## 1. Load Data

In [ ]:
train = pd.read_parquet(PROCESSED_DIR / 'train.parquet')
test  = pd.read_parquet(PROCESSED_DIR / 'test.parquet')
print(f'Train: {len(train):,}  Test: {len(test):,}')

## 2. Fit Popularity Model

In [ ]:
model = PopularityRecommender()
model.fit(train)
print('Top 10 popular movies:')
for rec in model.recommend(user_id=1, n=10, seen_items=set()):
    print(f'  movie_id={rec["movie_id"]}  score={rec["score"]:.0f}')

## 3. Evaluate

In [ ]:
ground_truth = build_ground_truth(test)
seen_items   = build_user_seen_items(train)

user_recs = {}
for uid in list(ground_truth.keys())[:500]:  # Sample for speed
    user_recs[uid] = [
        r['movie_id']
        for r in model.recommend(user_id=uid, n=10, seen_items=seen_items.get(uid, set()))
    ]

results = {}
for k in [5, 10, 20]:
    results[k] = compute_ranking_metrics(user_recs, ground_truth, k=k)
    print(f'K={k:2d}  P@K={results[k]["precision_at_k"]:.4f}  '
          f'HR@K={results[k]["hit_rate_at_k"]:.4f}  '
          f'NDCG@K={results[k]["ndcg_at_k"]:.4f}')

## 4. Visualise Metrics at Different K

In [ ]:
ks = [5, 10, 20]
metrics = ['precision_at_k', 'hit_rate_at_k', 'ndcg_at_k']
labels  = ['Precision@K', 'HR@K', 'NDCG@K']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, metric, label in zip(axes, metrics, labels):
    values = [results[k][metric] for k in ks]
    ax.bar(ks, values, width=4)
    ax.set_title(f'{label} (Popularity Baseline)')
    ax.set_xlabel('K')
    ax.set_xticks(ks)
    ax.set_ylim(0, max(values) * 1.3 if max(values) > 0 else 0.1)

plt.tight_layout()
plt.show()

model_logger.end_phase('baseline_nb', 'Baseline notebook complete')